# Cookie Model Training

Train ML models - 100k dataset

**Time:** 10-15 min with GPU

## Step 1: Upload Files

In [ ]:
from google.colab import files
import os

print('📤 Upload CSV files')
print('Select BOTH files:')
print('  - cookie_training_100k_train_80.csv')
print('  - cookie_training_100k_test_20.csv')

uploaded = files.upload()

if 'cookie_training_100k_train_80.csv' in uploaded and 'cookie_training_100k_test_20.csv' in uploaded:
    DATA_DIR = '/content'
    print(f'\n✅ Files uploaded!')
    print(f"✅ Training: {len(uploaded['cookie_training_100k_train_80.csv']):,} bytes")
    print(f"✅ Test: {len(uploaded['cookie_training_100k_test_20.csv']):,} bytes")
else:
    print('\n❌ Missing files!')

## Step 2: Install Dependencies

In [ ]:
!pip install -q scikit-learn xgboost imbalanced-learn pandas numpy matplotlib seaborn
print('✅ Dependencies installed!')

## Step 3: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
print('✅ Libraries imported!')

## Step 4: Load Data

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/cookie_training_100k_train_80.csv')
test_df = pd.read_csv(f'{DATA_DIR}/cookie_training_100k_test_20.csv')
print(f'📊 Train: {len(train_df):,} | Test: {len(test_df):,}')
train_df.head()

## Step 5: Remove Duplicates

In [ ]:
print('🔍 Checking for duplicates...')
train_before = len(train_df)
test_before = len(test_df)

# Remove duplicates based on cookie characteristics
train_df = train_df.drop_duplicates(subset=['cookie_name', 'cookie_domain', 'expiry_days', 'secure', 'http_only', 'is_third_party'])
test_df = test_df.drop_duplicates(subset=['cookie_name', 'cookie_domain', 'expiry_days', 'secure', 'http_only', 'is_third_party'])

train_after = len(train_df)
test_after = len(test_df)

print(f'✅ Train: {train_before:,} → {train_after:,} (removed {train_before-train_after:,} duplicates)')
print(f'✅ Test: {test_before:,} → {test_after:,} (removed {test_before-test_after:,} duplicates)')

## Step 6: Explore Data

In [ ]:
print('Class distribution:')
print(train_df['label_category'].value_counts())
plt.figure(figsize=(10, 6))
train_df['label_category'].value_counts().plot(kind='bar')
plt.title('Cookie Category Distribution')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 6: Prepare Features

In [ ]:
feature_columns = ['expiry_days', 'secure', 'http_only', 'is_third_party', 'cookie_name_len', 'is_ga_like', 'is_fb_like', 'has_prefix_utm', 'has_consent_kw', 'has_session_kw']
bool_columns = ['secure', 'http_only', 'is_third_party', 'is_ga_like', 'is_fb_like', 'has_prefix_utm', 'has_consent_kw', 'has_session_kw']
for col in bool_columns:
    train_df[col] = train_df[col].astype(int)
    test_df[col] = test_df[col].astype(int)
X_train = train_df[feature_columns].values
y_train = train_df['label_category'].values
X_test = test_df[feature_columns].values
y_test = test_df['label_category'].values
print(f'✅ Features: {X_train.shape} | Test: {X_test.shape}')

## Step 7: Encode Labels

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
print(f'✅ Categories: {list(label_encoder.classes_)}')

## Step 8: Create Validation Split

In [ ]:
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train_encoded, test_size=0.15, stratify=y_train_encoded, random_state=42)
print(f'📊 Train: {len(X_train_split):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

## Step 9: Balance Classes with SMOTE

In [ ]:
print('⚖️  Balancing classes...')
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_split, y_train_split)
print(f'✅ Balanced: {len(X_train_split):,} → {len(X_train_balanced):,}')

## Step 10: Train Random Forest

In [ ]:
print('🌲 Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=15, min_samples_leaf=8, max_features='sqrt', max_samples=0.8, class_weight='balanced', random_state=42, n_jobs=-1, verbose=1)
rf_model.fit(X_train_balanced, y_train_balanced)
val_acc_rf = accuracy_score(y_val, rf_model.predict(X_val))
print(f'✅ RF Validation Accuracy: {val_acc_rf:.4f}')

## Step 11: Train XGBoost

In [ ]:
print('🚀 Training XGBoost...')
xgb_model = xgb.XGBClassifier(objective='multi:softprob', num_class=len(label_encoder.classes_), max_depth=6, learning_rate=0.03, n_estimators=500, subsample=0.7, colsample_bytree=0.7, min_child_weight=5, gamma=0.2, reg_alpha=0.3, reg_lambda=2.0, random_state=42, tree_method='hist', eval_metric=['mlogloss', 'merror'], early_stopping_rounds=30)
xgb_model.fit(X_train_balanced, y_train_balanced, eval_set=[(X_train_balanced, y_train_balanced), (X_val, y_val)], verbose=True)
val_acc_xgb = accuracy_score(y_val, xgb_model.predict(X_val))
print(f'✅ XGB Validation Accuracy: {val_acc_xgb:.4f}')

## Step 12: Evaluate Models (FIXED)

In [ ]:
def eval_model(model, X, y, name):
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred, average='weighted', zero_division=0)
    print(f"\n{'='*70}")
    print(f"{name} - Test Results")
    print(f"{'='*70}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nPer-Class Metrics:")
    print(classification_report(y, y_pred, target_names=label_encoder.classes_, zero_division=0))
    cm = confusion_matrix(y, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'{name} - Confusion Matrix')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1}

# FIXED: Both lines now pass X_test (features) correctly!
rf_metrics = eval_model(rf_model, X_test, y_test_encoded, 'Random Forest')
xgb_metrics = eval_model(xgb_model, X_test, y_test_encoded, 'XGBoost')

## Step 13: Select Best Model

In [ ]:
print(f"\n{'='*70}")
print('Model Comparison')
print(f"{'='*70}\n")
comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'Accuracy': [rf_metrics['accuracy'], xgb_metrics['accuracy']],
    'F1-Score': [rf_metrics['f1_score'], xgb_metrics['f1_score']]
})
print(comparison_df.to_string(index=False))
print()
if xgb_metrics['f1_score'] > rf_metrics['f1_score']:
    best_model, best_name, best_type, best_metrics = xgb_model, 'XGBoost', 'xgboost', xgb_metrics
else:
    best_model, best_name, best_type, best_metrics = rf_model, 'Random Forest', 'random_forest', rf_metrics
print(f'🏆 Best Model: {best_name}')
print(f"   Accuracy: {best_metrics['accuracy']:.2%}")
print(f"   F1-Score: {best_metrics['f1_score']:.2%}")

## Step 14: Save Model

In [ ]:
output_dir = Path('/content/trained_models/v2.0.0')
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, output_dir / 'model.pkl')
joblib.dump(label_encoder, output_dir / 'label_encoder.pkl')
with open(output_dir / 'features.json', 'w') as f:
    json.dump(feature_columns, f, indent=2)
metadata = {'version': '2.0.0', 'model_type': best_type, 'trained_at': datetime.now().isoformat(), 'num_features': len(feature_columns), 'categories': list(label_encoder.classes_), 'training_samples': len(X_train), 'test_samples': len(X_test), 'metrics': best_metrics}
with open(output_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✅ Model saved to {output_dir}')

## Step 15: Download Model

In [ ]:
import shutil
shutil.make_archive('/content/trained_model_v2.0.0', 'zip', '/content/trained_models')
files.download('/content/trained_model_v2.0.0.zip')
print('✅ Download started!')

## Step 16: Test Predictions

In [ ]:
print('🧪 Testing with 5 random samples:\n')
for idx in np.random.choice(len(X_test), 5, replace=False):
    sample = X_test[idx:idx+1]
    true_label = label_encoder.classes_[y_test_encoded[idx]]
    pred_idx = best_model.predict(sample)[0]
    pred_label = label_encoder.classes_[pred_idx]
    confidence = best_model.predict_proba(sample)[0][pred_idx]
    cookie = test_df.iloc[idx]
    print(f"Cookie: {cookie['cookie_name']} | Domain: {cookie['cookie_domain']}")
    print(f"  True: {true_label} | Predicted: {pred_label} ({confidence:.1%}) {'✅' if pred_label==true_label else '❌'}\n")
print(f"\n🎉 Complete! Best: {best_name} | Accuracy: {best_metrics['accuracy']:.2%}")